# 05 — Prédire la victoire : modèle, équation, classement

**Règle d'or.** On n'utilise que des informations connues **avant** le match (sinon
c'est de la triche) : écart d'ELO, **forme récente** (net rating sur 5 matchs), derby.
Les facteurs du notebook 01 sont mesurés *pendant* le match → exclus de la prédiction.

**Plan.** (1) comparer plusieurs méthodes ; (2) construire le modèle de régression
logistique et son équation ; (3) prédire l'**écart de points** ; (4) prédire le
**classement final dès la mi-saison** (attendu du prof).

264 matchs (saisons 24-25 + 25-26, calendrier officiel).

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "analyse" / "outils.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p / "analyse"))
import numpy as np, pandas as pd, outils as o
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
OUT = o.RESULTATS / "prediction"; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)
print("sorties ->", OUT)

sorties -> C:\Users\TEMP.IUT-LUMIERE.000\R6.06-Domaines-d-application-de-la-statistique\analyse\resultats\prediction


In [2]:
cal = o.charger_calendrier(forme=True)
feats = ["d_elo","d_forme","derby"]
X = cal[feats].values; y = cal["home_win"].values
print(f"{len(cal)} matchs | victoires a domicile : {y.mean()*100:.0f}%")

264 matchs | victoires a domicile : 60%


## 1. Comparaison des méthodes (validation croisée 5-fold)

In [3]:
def ev(model, Xx, nom):
    auc = cross_val_score(model, Xx, y, cv=5, scoring="roc_auc").mean()
    acc = cross_val_score(model, Xx, y, cv=5, scoring="accuracy").mean()
    return {"methode":nom, "AUC":round(auc,3), "accuracy_%":round(acc*100,1)}

Xs  = StandardScaler().fit_transform(X)
Xe  = StandardScaler().fit_transform(cal[["d_elo"]].values)
res = pd.DataFrame([
    {"methode":"baseline (toujours domicile)", "AUC":0.500, "accuracy_%":round(max(y.mean(),1-y.mean())*100,1)},
    ev(LogisticRegression(max_iter=2000), Xe, "ELO seul (logistique)"),
    ev(LogisticRegression(max_iter=2000), Xs, "Logistique (ELO+forme+derby)"),
    ev(RandomForestClassifier(n_estimators=400,max_depth=4,min_samples_leaf=8,random_state=0), X, "Random Forest"),
    ev(GradientBoostingClassifier(n_estimators=200,max_depth=2,learning_rate=0.05,random_state=0), X, "Gradient Boosting"),
])
print(res.to_string(index=False))

                     methode   AUC  accuracy_%
baseline (toujours domicile) 0.500        59.8
       ELO seul (logistique) 0.750        66.7
Logistique (ELO+forme+derby) 0.761        69.7
               Random Forest 0.743        68.6
           Gradient Boosting 0.686        66.7


**Lecture.** La **régression logistique (ELO + forme)** est le meilleur compromis
performance/lisibilité. La Random Forest ne fait pas mieux en AUC : avec ~260 matchs et
peu de variables, **le modèle simple gagne**. La forme récente est le 2ᵉ facteur après
l'ELO (le derby et l'historique sont marginaux — vu au notebook 02).

## 2. Le modèle probabiliste et son équation

On récupère les coefficients sur l'échelle **réelle** des variables (pas standardisée)
pour une équation lisible et interprétable.

In [4]:
sc = StandardScaler().fit(X)
clf = LogisticRegression(max_iter=2000).fit(sc.transform(X), y)
proba = cross_val_predict(clf, sc.transform(X), y, cv=5, method="predict_proba")[:,1]
print(f"AUC={roc_auc_score(y,proba):.3f}  accuracy={accuracy_score(y,(proba>=.5).astype(int))*100:.1f}%")
print(f"Brier score={brier_score_loss(y,proba):.3f} (reference 'toujours 50%' = 0.250)")

b  = clf.coef_[0] / sc.scale_
b0 = clf.intercept_[0] - (clf.coef_[0]*sc.mean_/sc.scale_).sum()
print("\nlogit(P_victoire_domicile) =")
print(f"  {b0:+.3f}")
for f_,c_ in zip(feats, b):
    print(f"  {c_:+.5f} x {f_}")
print(f"\nAvantage du terrain pur (tout a 0) : P = {1/(1+np.exp(-b0)):.2f}")

AUC=0.760  accuracy=69.7%
Brier score=0.194 (reference 'toujours 50%' = 0.250)

logit(P_victoire_domicile) =
  +0.610
  +0.00471 x d_elo
  +0.03647 x d_forme
  -0.59856 x derby

Avantage du terrain pur (tout a 0) : P = 0.65


**Lecture de l'équation.** Le logit (= log des cotes) est modélisé linéairement, puis
converti en probabilité par `P = 1/(1+e^-logit)`. Avantage du terrain ≈ **65 %** à forces
égales ; **+100 ELO ≈ +10 points de proba** ; le **derby** (coef négatif) annule
l'avantage du terrain — les derbys sont indécis.

In [5]:
cal2 = cal.assign(proba=proba)
rel = (cal2.groupby(pd.cut(cal2["proba"], [0,.35,.5,.65,.8,1.0]), observed=False)
          .agg(n=("home_win","size"), predit=("proba","mean"), observe=("home_win","mean")))
cal2.to_csv(OUT / "matchs_proba.csv", index=False)
rel["predit"] = (rel["predit"]*100).round(0); rel["observe"] = (rel["observe"]*100).round(0)
print("Calibration (predit vs observe, en %) :")
print(rel.to_string())

Calibration (predit vs observe, en %) :
              n  predit  observe
proba                           
(0.0, 0.35]  44    26.0     30.0
(0.35, 0.5]  48    43.0     42.0
(0.5, 0.65]  51    58.0     51.0
(0.65, 0.8]  59    72.0     75.0
(0.8, 1.0]   62    87.0     89.0


**Calibration.** « prédit » ≈ « observé » sur toute l'échelle → les probabilités sont
**honnêtes** (quand le modèle dit 70 %, l'équipe gagne ~70 % du temps).

## 3. Prédire l'écart de points (régression linéaire)

In [6]:
ye = cal["ecart"].values
mae_base = np.mean(np.abs(ye - ye.mean()))
mae_elo  = -cross_val_score(LinearRegression(), cal[["d_elo"]], ye, cv=5, scoring="neg_mean_absolute_error").mean()
r2_elo   = cross_val_score(LinearRegression(), cal[["d_elo"]], ye, cv=5, scoring="r2").mean()
lr = LinearRegression().fit(cal[["d_elo"]], ye)
print(f"baseline (ecart moyen)      MAE={mae_base:.1f} pts")
print(f"lineaire (ELO)              MAE={mae_elo:.1f} pts  R2={r2_elo:+.2f}")
print(f"\nCalibration : +100 ELO d'avance = {lr.coef_[0]*100:+.1f} pts d'ecart attendus "
      f"(+ {lr.intercept_:+.1f} pts d'avantage terrain)")

baseline (ecart moyen)      MAE=11.5 pts
lineaire (ELO)              MAE=10.2 pts  R2=+0.20

Calibration : +100 ELO d'avance = +4.4 pts d'ecart attendus (+ +2.9 pts d'avantage terrain)


On prédit l'écart à **±10 points** : honnête mais imprécis (écart-type réel ~14.6 pts).
Une grande part de la marge est de l'aléa irréductible.

## 4. Classement final prédit dès la mi-saison (journée 11)

Pour chaque saison : victoires acquises (J1-11) + **espérance de victoires** sur les
matchs restants (J12-22) via le modèle → total → classement, comparé au réel.

In [7]:
MI = 11
for saison in sorted(cal["Saison"].unique()):
    s = cal[cal["Saison"]==saison]; equipes = sorted(set(s["dom"])|set(s["ext"]))
    acquis = {e:0.0 for e in equipes}
    for _,r in s[s["jour"]<=MI].iterrows():
        acquis[r["dom"] if r["home_win"] else r["ext"]] += 1
    esper = {e:0.0 for e in equipes}; rest = s[s["jour"]>MI]
    pr = clf.predict_proba(sc.transform(rest[feats].values))[:,1]
    for (_,r),ph in zip(rest.iterrows(), pr):
        esper[r["dom"]] += ph; esper[r["ext"]] += 1-ph
    pred = {e:acquis[e]+esper[e] for e in equipes}
    reel = {e:0 for e in equipes}
    for _,r in s.iterrows(): reel[r["dom"] if r["home_win"] else r["ext"]] += 1
    prank = sorted(equipes, key=lambda e:-pred[e]); rrank = sorted(equipes, key=lambda e:-reel[e])
    posr = {e:i+1 for i,e in enumerate(rrank)}
    tab = pd.DataFrame({"rang_pred":range(1,len(equipes)+1), "equipe":prank,
            "V_pred":[round(pred[e],1) for e in prank], "V_reel":[reel[e] for e in prank],
            "rang_reel":[posr[e] for e in prank]})
    tab["ecart"] = tab["rang_pred"]-tab["rang_reel"]
    tab.to_csv(OUT / f"classement_{saison}.csv", index=False)
    err = tab["ecart"].abs().mean()
    rho = np.corrcoef([prank.index(e) for e in equipes],[rrank.index(e) for e in equipes])[0,1]
    print(f"=== Saison {saison} : erreur moyenne {err:.1f} rang, correlation {rho:.2f}, "
          f"champion '{prank[0]}' vs reel '{rrank[0]}' {'OK' if prank[0]==rrank[0] else 'X'} ===")
    print(tab.to_string(index=False)); print()

=== Saison 24-25 : erreur moyenne 0.7 rang, correlation 0.96, champion 'bourges' vs reel 'bourges' OK ===
 rang_pred          equipe  V_pred  V_reel  rang_reel  ecart
         1         bourges    17.6      19          1      0
         2     montpellier    14.8      15          4     -2
         3         charnay    14.7      15          3      0
         4   basket landes    13.3      15          2      2
         5     charleville    12.7      13          5      0
         6            lyon    11.8      11          7     -1
         7          angers    11.5      12          6      1
         8          tarbes    10.8       9          8      0
         9 villeneuve ascq     9.1       7         10     -1
        10      landerneau     6.7       7          9      1
        11    roche vendee     5.5       6         11      0
        12        chartres     3.6       3         12      0

=== Saison 25-26 : erreur moyenne 1.0 rang, correlation 0.91, champion 'basket landes' vs reel 'bask

## Conclusion

- **Outil opérationnel** : probabilité calibrée par match + classement prévisionnel fiable
  dès la J11 (erreur < 1 rang, **champion correct les deux saisons**).
- **Le niveau (ELO) fait l'essentiel**, la forme affine, le reste est marginal.
- **Limite** : le milieu de tableau reste incertain (matchs serrés) ; haut et bas très
  prévisibles. Résultats dans `resultats/prediction/`.